In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# ================================
#  Baseline: ElasticNet + Purged Time CV + EWMA sizing
#  - Dict-based configs (no dataclass)
#  - Auto-detects train/test under /kaggle/input
#  - Produces /kaggle/working/allocations.csv
# ================================
from pathlib import Path
import os
import numpy as np
import pandas as pd
from typing import Optional, Tuple, List

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from inspect import signature

import lightgbm as lgb
import xgboost as xgb


/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [3]:
# =========================
# Config / Toggles
# =========================
FAST_SUBMIT = True            # keep True for Kaggle "Code" runs to avoid timeout
USE_XGB     = True            # set False if you want LGB only
USE_GPU_XGB = True            # you said you use GPU -> keep True

# CV & seeds
if FAST_SUBMIT:
    NFOLDS = 5
    SEEDS  = [42]
else:
    NFOLDS = 10
    SEEDS  = [42, 2025, 7]

EARLY_STOP_ROUNDS = 100

# Feature options
LAGS         = [1, 5, 20]
ROLLS        = [5, 20]
ADD_CALENDAR = True     # adds DoW/Month if date exists

# Debias toggle
TRY_LINEAR_DEBIAS = True

# Output
OUT_PARQ = Path("/kaggle/working/submission.parquet")

# =========================
# Column guesses
# =========================
DEFAULT_DATE_GUESSES  = ["date", "Date", "timestamp", "Timestamp", "datetime", "Datetime", "date_id"]
DEFAULT_GROUP_GUESSES = ["symbol", "ticker", "asset", "asset_id"]
DEFAULT_ID_GUESSES    = ["id", "row_id", "date_id", "time_id"]

TARGET    = None   # if None, auto-pick a numeric col (not id/date/group)
DATE_COL  = None   # if None, guess from DEFAULT_DATE_GUESSES
GROUP_COL = None   # if None, guess from DEFAULT_GROUP_GUESSES (panel)

# =========================
# Discover input files under /kaggle/input
# =========================
train_path = test_path = sample_path = None
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        fname = filename.lower()
        full = os.path.join(dirname, filename)
        if fname == "train.csv":
            train_path = Path(full)
        elif fname == "test.csv":
            test_path = Path(full)
        elif "sample" in fname and "submission" in fname and fname.endswith(".csv"):
            sample_path = Path(full)

if not train_path or not test_path:
    raise FileNotFoundError("Could not find train.csv or test.csv under /kaggle/input")

TRAIN_CSV  = train_path
TEST_CSV   = test_path
SAMPLE_SUB = sample_path
DATA_DIR   = TRAIN_CSV.parent

# Determine ID & PRED names
ID_COL, PRED_COL = "id", "prediction"
if SAMPLE_SUB and SAMPLE_SUB.exists():
    _ss = pd.read_csv(SAMPLE_SUB)
    if _ss.shape[1] >= 2:
        ID_COL, PRED_COL = _ss.columns[:2].tolist()
else:
    # Fallback guess if sample not provided
    # Prefer *_id style column if present
    test_head = pd.read_csv(TEST_CSV, nrows=1)
    lower_map = {c.lower(): c for c in test_head.columns}
    for guess in DEFAULT_ID_GUESSES:
        if guess in lower_map:
            ID_COL = lower_map[guess]
            break
    # prediction col name if comp doesn't give one
    PRED_COL = PRED_COL or "prediction"

print(f"DATA_DIR   = {DATA_DIR}")
print(f"TRAIN_CSV  = {TRAIN_CSV}")
print(f"TEST_CSV   = {TEST_CSV}")
print(f"SAMPLE_SUB = {SAMPLE_SUB}")
print(f"ID_COL={ID_COL}  PRED_COL={PRED_COL}")

# =========================
# Helpers
# =========================
def rmse(y, p):
    return mean_squared_error(y, p, squared=False)

def annualized_sharpe(series: np.ndarray, period_per_year=252):
    eps = 1e-12
    mu = float(np.mean(series))
    sd = float(np.std(series) + eps)
    return (mu / sd) * np.sqrt(period_per_year)

def analytic_weight(a, b, y):
    a = a.astype(float); b = b.astype(float); y = y.astype(float)
    num = np.dot(y - b, a - b)
    den = np.dot(a - b, a - b) + 1e-12
    return float(np.clip(num / den, 0.0, 1.0))

def _guess_col(cols: List[str], guesses: List[str]) -> Optional[str]:
    low = {c.lower(): c for c in cols}
    for g in guesses:
        if g.lower() in low:
            return low[g.lower()]
    return None

def _autodetect(train: pd.DataFrame) -> Tuple[str, Optional[str], Optional[str], str]:
    global TARGET, DATE_COL, GROUP_COL, ID_COL, PRED_COL
    date_col  = DATE_COL or _guess_col(train.columns.tolist(), DEFAULT_DATE_GUESSES)
    group_col = GROUP_COL or _guess_col(train.columns.tolist(), DEFAULT_GROUP_GUESSES)
    # target
    if TARGET is None:
        num_cols = [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c])]
        candidates = [c for c in num_cols if c not in {ID_COL, date_col, group_col}]
        target = candidates[-1] if candidates else None
    else:
        target = TARGET
    if target is None:
        raise ValueError("Could not auto-detect TARGET. Please set TARGET explicitly.")
    return target, date_col, group_col, ID_COL

def _ensure_datetime(df: pd.DataFrame, date_col: Optional[str]):
    if date_col is None:
        return df, None
    if not np.issubdtype(df[date_col].dtype, np.datetime64):
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df, date_col

DATA_DIR   = /kaggle/input/hull-tactical-market-prediction
TRAIN_CSV  = /kaggle/input/hull-tactical-market-prediction/train.csv
TEST_CSV   = /kaggle/input/hull-tactical-market-prediction/test.csv
SAMPLE_SUB = None
ID_COL=date_id  PRED_COL=prediction


In [4]:
# =========================
# CLEANING (time-series-safe)
# =========================
def basic_clean(train, test, id_col, target, date_col=None, group_col=None):
    # 1) Parse dates
    if date_col is not None:
        for df in (train, test):
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # 2) Drop exact duplicates
    train = train.drop_duplicates().reset_index(drop=True)
    test  = test.drop_duplicates().reset_index(drop=True)

    # 3) Enforce ID uniqueness in test
    if id_col in test.columns:
        assert test[id_col].is_unique, "Test IDs must be unique."

    # 4) Drop rows with missing target
    if target in train.columns:
        train = train[train[target].notna()].reset_index(drop=True)

    # 5) Sort (panel-safe): by date first, then group (if provided)
    sort_cols = [c for c in [date_col, group_col] if c is not None]
    if sort_cols:
        train = train.sort_values(sort_cols).reset_index(drop=True)
        test  = test.sort_values(sort_cols).reset_index(drop=True)

    # 6) Coerce numeric-like columns (leave categoricals as is)
    def _coerce_numeric(df):
        for c in df.columns:
            if c in {id_col, target, date_col, group_col}:
                continue
            # try numeric; if fails, leave as-is (trees can handle category if encoded)
            if df[c].dtype == "object":
                df[c] = pd.to_numeric(df[c], errors="ignore")
        return df
    train = _coerce_numeric(train)
    test  = _coerce_numeric(test)

    # 7) Drop ultra-missing columns (>99% NA in train)
    miss_ratio = train.isna().mean(numeric_only=False)
    drop_cols = miss_ratio[miss_ratio > 0.99].index.tolist()
    if drop_cols:
        train = train.drop(columns=drop_cols)
        test  = test.drop(columns=[c for c in drop_cols if c in test.columns])

    # 8) Per-group forward/back fill for FEATURES (NOT target)
    base_feat_cols = [c for c in train.columns if c not in {id_col, target, date_col, group_col}]
    def _ffill_block(df):
        feat_cols = [c for c in base_feat_cols if c in df.columns]
        if not feat_cols:
            return df
        if group_col and group_col in df.columns:
            df[feat_cols] = df.groupby(group_col, group_keys=False)[feat_cols].apply(lambda g: g.ffill().bfill())
        else:
            df[feat_cols] = df[feat_cols].ffill().bfill()
        return df
    train = _ffill_block(train)
    test  = _ffill_block(test)

    # 9) Remaining NA -> 0 (OK for trees)
    feat_cols_train = [c for c in train.columns if c not in {id_col, target, date_col, group_col}]
    feat_cols_test  = [c for c in test.columns  if c not in {id_col, target, date_col, group_col}]
    train[feat_cols_train] = train[feat_cols_train].fillna(0.0)
    test[feat_cols_test]   = test[feat_cols_test].fillna(0.0)

    # 10) Remove zero-variance columns (on train; apply to test if exists)
    nunique = train[feat_cols_train].nunique(dropna=False)
    zvars = nunique[nunique <= 1].index.tolist()
    if zvars:
        train = train.drop(columns=zvars)
        test  = test.drop(columns=[c for c in zvars if c in test.columns])

    # 11) Gentle winsorization (0.1%–99.9%) on numeric (non-bool) features present in both
    def _is_num_not_bool(s: pd.Series) -> bool:
        return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

    common_feats = [c for c in train.columns if c in test.columns and c not in {id_col, target, date_col, group_col}]
    num_feats    = [c for c in common_feats if _is_num_not_bool(train[c])]
    if num_feats:
        qlo = train[num_feats].quantile(0.001)
        qhi = train[num_feats].quantile(0.999)
        train[num_feats] = train[num_feats].clip(lower=qlo, upper=qhi, axis=1)
        test[num_feats]  = test[num_feats].clip(lower=qlo, upper=qhi, axis=1)

    return train.reset_index(drop=True), test.reset_index(drop=True)

In [5]:
# =========================
# Time-safe Feature Engineering (panel-aware)
# =========================
def build_features(df: pd.DataFrame, id_col: str, target: str, date_col: Optional[str], group_col: Optional[str]) -> pd.DataFrame:
    out = df.copy()
    if date_col is not None:
        sort_cols = [date_col] + ([group_col] if group_col and group_col in out.columns else [])
        out = out.sort_values(sort_cols)

    # base numeric predictors (exclude id/target)
    base_num = [c for c in out.columns if pd.api.types.is_numeric_dtype(out[c])]
    base_num = [c for c in base_num if c not in {id_col, target}]

    def _panel_apply(g):
        g = g.copy()
        for col in base_num:
            for L in LAGS:
                g[f"{col}_lag{L}"] = g[col].shift(L)
            for W in ROLLS:
                g[f"{col}_rollmean{W}"] = g[col].shift(1).rolling(W, min_periods=max(2, int(W*0.6))).mean()
                g[f"{col}_rollstd{W}"]  = g[col].shift(1).rolling(W, min_periods=max(2, int(W*0.6))).std()
                g[f"{col}_z{W}"] = (g[col].shift(1) - g[f"{col}_rollmean{W}"]) / (g[f"{col}_rollstd{W}"] + 1e-12)
        return g

    if group_col and group_col in out.columns:
        out = out.groupby(group_col, group_keys=False).apply(_panel_apply)
    else:
        out = _panel_apply(out)

    if date_col and ADD_CALENDAR:
        out["dow"]   = out[date_col].dt.weekday
        out["month"] = out[date_col].dt.month

    # Fill minimal to ensure numeric matrices
    out = out.fillna(method="ffill")
    out = out.fillna(0.0)
    return out


In [6]:
# =========================
# Models (GPU-safe XGB 2.x)
# =========================
def train_lgb(X_tr, y_tr, X_va, y_va, seed, variant="A"):
    if variant == "A":
        params = dict(
            objective="rmse",
            learning_rate=0.03,
            n_estimators=5000,
            feature_fraction=0.85,
            bagging_fraction=0.85,
            bagging_freq=1,
            num_leaves=255,
            min_child_samples=20,
            max_bin=255,
            lambda_l2=1.0,
            random_state=seed,
            n_jobs=-1,
            force_col_wise=True,
        )
    else:
        params = dict(
            objective="rmse",
            learning_rate=0.03,
            n_estimators=5000,
            feature_fraction=0.85,
            bagging_fraction=0.85,
            bagging_freq=1,
            num_leaves=191,
            min_child_samples=40,
            max_bin=255,
            lambda_l2=1.0,
            random_state=seed,
            n_jobs=-1,
            force_col_wise=True,
        )
    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS), lgb.log_evaluation(0)]
    )
    return m

def train_xgb(X_tr, y_tr, X_va, y_va, seed, use_gpu=False, variant="A"):
    if variant == "A":
        depth, mcw, gamma, col = 7, 3, 0.10, 0.80
    else:
        depth, mcw, gamma, col = 6, 4, 0.15, 0.80

    device = "cuda" if use_gpu else "cpu"
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        learning_rate=0.03,
        n_estimators=5000,
        max_depth=depth,
        subsample=0.8,
        colsample_bytree=col,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=mcw,
        gamma=gamma,
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",    # keep 'hist'; GPU is controlled by device=...
        device=device,         # XGBoost 2.x way to use GPU
        eval_metric="rmse",
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        early_stopping_rounds=EARLY_STOP_ROUNDS,
        verbose=False
    )
    return model

# =========================
# Fold runner (TimeSeriesSplit; panel safe if sorted by date)
# =========================
def run_combo(X, y, T, lgb_variant="A", xgb_variant="A"):
    n = len(y); m = len(T)
    oof_lgb = np.zeros(n, dtype=np.float32); te_lgb = np.zeros(m, dtype=np.float32)
    oof_xgb = np.zeros(n, dtype=np.float32); te_xgb = np.zeros(m, dtype=np.float32)

    fold_slices = []
    oof_lgb_folds, oof_xgb_folds, y_folds = [], [], []

    for seed in SEEDS:
        tscv = TimeSeriesSplit(n_splits=NFOLDS)

        # LGB
        oof_tmp = np.zeros(n, dtype=np.float32); te_tmp  = np.zeros(m, dtype=np.float32)
        for f, (tr, va) in enumerate(tscv.split(X), 1):
            mdl = train_lgb(X.iloc[tr], y[tr], X.iloc[va], y[va], seed, variant=lgb_variant)
            va_pred = mdl.predict(X.iloc[va], num_iteration=getattr(mdl, "best_iteration_", None)).astype("float32")
            te_pred = mdl.predict(T,        num_iteration=getattr(mdl, "best_iteration_", None)).astype("float32")
            oof_tmp[va] = va_pred
            te_tmp     += te_pred / NFOLDS
            if (seed == SEEDS[0]):
                fold_slices.append(va)
        print(f"[seed {seed}] LGB({lgb_variant}) OOF RMSE: {rmse(y, oof_tmp):.6f}")
        oof_lgb += oof_tmp / len(SEEDS)
        te_lgb  += te_tmp  / len(SEEDS)

        # XGB (optional)
        if USE_XGB:
            oof_tmp = np.zeros(n, dtype=np.float32); te_tmp  = np.zeros(m, dtype=np.float32)
            tscv2 = TimeSeriesSplit(n_splits=NFOLDS)
            for f, (tr, va) in enumerate(tscv2.split(X), 1):
                mdl = train_xgb(X.iloc[tr], y[tr], X.iloc[va], y[va], seed, use_gpu=USE_GPU_XGB, variant=xgb_variant)
                va_pred = mdl.predict(X.iloc[va]).astype("float32")
                te_pred = mdl.predict(T).astype("float32")
                oof_tmp[va] = va_pred
                te_tmp     += te_pred / NFOLDS
            print(f"[seed {seed}] XGB({xgb_variant}) OOF RMSE: {rmse(y, oof_tmp):.6f}")
            oof_xgb += oof_tmp / len(SEEDS)
            te_xgb  += te_tmp  / len(SEEDS)

    # per-fold collections using averaged OOF
    oof_lgb_folds = [oof_lgb[idx] for idx in fold_slices]
    if USE_XGB:
        oof_xgb_folds = [oof_xgb[idx] for idx in fold_slices]
    else:
        oof_xgb_folds = [np.zeros_like(oof_lgb[idx]) for idx in fold_slices]
    y_folds       = [y[idx] for idx in fold_slices]

    return oof_lgb, te_lgb, oof_xgb, te_xgb, oof_lgb_folds, oof_xgb_folds, y_folds, fold_slices

In [7]:
# =========================
# Main
# =========================
def main():
    assert TRAIN_CSV.exists() and TEST_CSV.exists(), "CSV files not found at Kaggle input path"
    train = pd.read_csv(TRAIN_CSV)
    test  = pd.read_csv(TEST_CSV)

    # Auto-detect columns
    target, date_col, group_col, id_col = _autodetect(train)
    print(f"Detected -> TARGET={target}  DATE_COL={date_col}  GROUP_COL={group_col}  ID_COL={id_col}  PRED_COL={PRED_COL}")

    # Ensure datetime; sort
    train, date_col = _ensure_datetime(train, date_col)
    test,  _        = _ensure_datetime(test,  date_col)

    # CLEAN (time-safe)
    train, test = basic_clean(train, test, id_col=id_col, target=target, date_col=date_col, group_col=group_col)

    # Concat for time-safe FE so test rolls can use train history
    train["__is_train__"] = 1
    test["__is_train__"]  = 0
    both = pd.concat([train, test], axis=0, ignore_index=True)

    # Build features (strictly lag/rolling)
    both_fe = build_features(both, id_col=id_col, target=target, date_col=date_col, group_col=group_col)

    # Split back
    train_fe = both_fe[both_fe["__is_train__"] == 1].drop(columns=["__is_train__"])
    test_fe  = both_fe[both_fe["__is_train__"] == 0].drop(columns=["__is_train__"])

    # Feature columns
    drop_cols = {id_col, target}
    if date_col:  drop_cols.add(date_col)
    if group_col: drop_cols.add(group_col)
    feat_cols = [c for c in train_fe.columns if c not in drop_cols]

    # Prepare X,y,T
    y = train_fe[target].values.astype("float32")
    X = train_fe[feat_cols].astype("float32")
    T = test_fe[feat_cols].astype("float32")
    print(f"Shapes -> X:{X.shape}  T:{T.shape}  y:{len(y)}  feats:{len(feat_cols)}")

    # A/B model combos
    combos = [("A","A")]  # you can expand: [("A","A"),("B","A"),("A","B"),("B","B")]
    best = None

    for (lgb_v, xgb_v) in combos:
        print(f"\n=== Running combo LGB({lgb_v}) + XGB({xgb_v}) ===")
        (oof_lgb, te_lgb,
         oof_xgb, te_xgb,
         oof_lgb_folds, oof_xgb_folds, y_folds, fold_slices) = run_combo(X, y, T, lgb_v, xgb_v)

        # Blend 1: global analytic weight (if XGB disabled, w=1 -> LGB only)
        if USE_XGB:
            w_global = analytic_weight(oof_lgb, oof_xgb, y)
        else:
            w_global = 1.0
        oof_blend_global = w_global * oof_lgb + (1 - w_global) * oof_xgb
        rmse_global = rmse(y, oof_blend_global)
        te_blend_global = w_global * te_lgb + (1 - w_global) * te_xgb

        # Blend 2: per-fold analytic weights -> mean weight applied to test
        weights_f = []
        oof_stack = np.zeros_like(y, dtype=float)
        for idx, a_f, b_f, y_f in zip(fold_slices, oof_lgb_folds, oof_xgb_folds, y_folds):
            w_f = analytic_weight(a_f, b_f, y_f) if USE_XGB else 1.0
            weights_f.append(w_f)
            oof_stack[idx] = w_f * a_f + (1 - w_f) * b_f
        w_mean = float(np.mean(weights_f)) if len(weights_f) else 1.0
        rmse_pf = rmse(y, oof_stack)
        te_blend_pf = w_mean * te_lgb + (1 - w_mean) * te_xgb

        if rmse_pf + 1e-12 < rmse_global:
            blend_name = f"per-fold (mean w={w_mean:.4f})"
            oof_blend = oof_stack
            te_blend  = te_blend_pf
            score_blend = rmse_pf
            w_used = w_mean
        else:
            blend_name = f"global (w={w_global:.4f})"
            oof_blend = oof_blend_global
            te_blend  = te_blend_global
            score_blend = rmse_global
            w_used = w_global

        # Diagnostics
        r2 = r2_score(y, oof_blend)
        sr = annualized_sharpe(oof_blend)
        print(f"Blend: {blend_name} | OOF RMSE: {score_blend:.6f} | OOF R2: {r2:.6f} | OOF Sharpe-proxy: {sr:.4f}")

        # Optional linear debias
        applied_debias = False
        if TRY_LINEAR_DEBIAS:
            lr = LinearRegression().fit(oof_blend.reshape(-1,1), y)
            oof_cal = lr.predict(oof_blend.reshape(-1,1))
            rmse_cal = rmse(y, oof_cal)
            r2_cal   = r2_score(y, oof_cal)
            sr_cal   = annualized_sharpe(oof_cal)
            print(f"Linear debias: slope={lr.coef_[0]:.6f} intercept={lr.intercept_:.6f} "
                  f"| OOF RMSE={rmse_cal:.6f} R2={r2_cal:.6f} Sharpe={sr_cal:.4f}")
            if rmse_cal + 1e-12 < score_blend:
                te_blend = lr.predict(te_blend.reshape(-1,1))
                oof_blend = oof_cal
                score_blend = rmse_cal
                applied_debias = True
                print("Debias improved OOF — applied to test.")
            else:
                print("Debias did not help — keeping raw blend.")

        result = dict(
            lgb_variant=lgb_v, xgb_variant=xgb_v,
            w_used=w_used, blend_mode=blend_name,
            debias=applied_debias, oof_rmse=score_blend,
            oof_r2=r2_score(y, oof_blend), oof_sharpe=annualized_sharpe(oof_blend),
            te_pred=te_blend.astype("float32")
        )
        if (best is None) or (score_blend + 1e-12 < best["oof_rmse"]):
            best = result

    print("\n=== BEST CONFIG ===")
    print(best)

 # Build submission aligned to sample or detected ID
    assert ID_COL in test.columns, f"ID column '{ID_COL}' not in test.csv. Found: {list(test.columns)[:10]}..."
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, PRED_COL: best["te_pred"]})

    # Write parquet (Kaggle expects /kaggle/working/submission.parquet)
    OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
    sub.to_parquet(OUT_PARQ, index=False, engine="pyarrow")
    print(f"Wrote {OUT_PARQ}  shape={sub.shape}")

if __name__ == "__main__":
    main()

Detected -> TARGET=market_forward_excess_returns  DATE_COL=date_id  GROUP_COL=None  ID_COL=date_id  PRED_COL=prediction


/tmp/ipykernel_36/1637574176.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  g[f"{col}_rollmean{W}"] = g[col].shift(1).rolling(W, min_periods=max(2, int(W*0.6))).mean()
/tmp/ipykernel_36/1637574176.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  g[f"{col}_rollstd{W}"]  = g[col].shift(1).rolling(W, min_periods=max(2, int(W*0.6))).std()
/tmp/ipykernel_36/1637574176.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

Shapes -> X:(8990, 1002)  T:(10, 1002)  y:8990  feats:1002

=== Running combo LGB(A) + XGB(A) ===
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fr

/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [01:46:24] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early

[seed 42] XGB(A) OOF RMSE: 0.008911
Blend: per-fold (mean w=1.0000) | OOF RMSE: 0.003225 | OOF R2: 0.906887 | OOF Sharpe-proxy: 0.0276
Linear debias: slope=1.061360 intercept=0.000033 | OOF RMSE=0.003171 R2=0.909939 Sharpe=0.0796
Debias improved OOF — applied to test.

=== BEST CONFIG ===
{'lgb_variant': 'A', 'xgb_variant': 'A', 'w_used': 1.0, 'blend_mode': 'per-fold (mean w=1.0000)', 'debias': True, 'oof_rmse': 0.0031712285907371457, 'oof_r2': 0.9099386520468224, 'oof_sharpe': 0.07960079256189279, 'te_pred': array([-0.00655271, -0.00825305, -0.00817005,  0.00779488, -0.00364632,
       -0.00347654,  0.00174183,  0.00234078,  0.00782862, -0.00051054],
      dtype=float32)}
Wrote /kaggle/working/submission.parquet  shape=(10, 2)
